# Eye - ADEM_TEST, own method [OURS]
**Spec:** PLAN.md B2, FINDINGS 1. **Runs on:** Colab. Pilot run 2026-10-12; export cells at the end added after.

Upload these two files (made on your PC in `D:\Alzheimers\colab_upload\`) to the Colab Files panel (`/content`):

| File | Size |
|---|---|
| `eye_code.tar` (`common/`, `eye/`) | tiny |
| `eye_data.tar` (ADEM_TEST: `AD/`, `NC/`, `MMSE_MOCA.xlsx`) | ~35 MB |

Re-upload both in every new session. Pilot: 20 subjects, leave-one-subject-out. Report only what this prints.

In [ ]:
# Run in EVERY new session: unpack what you uploaded into /content
import os, sys, subprocess
os.makedirs('/content/Alzheimers', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
subprocess.run(['tar', '-xf', '/content/eye_code.tar', '-C', '/content/Alzheimers'], check=True)
subprocess.run(['tar', '-xf', '/content/eye_data.tar', '-C', '/content/data'], check=True)   # creates /content/data/ADEM_TEST
import numpy as np, pandas as pd
REPO = '/content/Alzheimers'          # folder containing common/ and eye/
DATA = '/content/data/ADEM_TEST'      # folder holding AD/ and NC/
sys.path.insert(0, REPO)
from eye.features import build_table, model_columns, TASKS
from eye.cv import evaluate, loso
from common.runlog import RunLog
log = RunLog('/content/out/eye.log')   # appends to a text file on Colab's temporary disk


## 1. Features per subject

In [ ]:
with log.tee('features'):
    df = pd.DataFrame(build_table(DATA))
    print(df.shape, df.label.value_counts().to_dict())
df.filter(regex='subject|label|missing_frac|n_sac').round(2)


> **SEND BACK TO CLAUDE:**
> the printed shape and label counts line (expect 20 subjects). The table itself is optional.

## 2. LOSO (all 3 tasks together, 0.5 cut-off)

In [ ]:
with log.tee('LOSO'):
    cols = model_columns()
    for r in evaluate(df[cols].values, df.label.values): print(r)


> **SEND BACK TO CLAUDE:**
> the printed result lines (balanced accuracy and AUC, with CI, n, protocol).

## 3. Per task (context only; tasks were fixed in advance, nothing is selected from this)

In [ ]:
with log.tee('per task'):
    for t in TASKS:
        c = [k for k in cols if k.startswith(t + '.')]
        for r in evaluate(df[c].values, df.label.values): print(t, '|', r)


Check before reporting: how many recordings have no saccade at all (features imputed), and that `missing_frac` was not given to the model (it is a diagnostic).

## Detector check (bug check, NOT tuning)
Why so few saccades? For every recording this prints the sampling rate, the velocity threshold the detector chose, and whether that threshold is *above the largest velocity in the recording* (then zero saccades are possible by construction). Nothing here changes the model or the results above.

In [ ]:
with log.tee('detector check'):
    from pathlib import Path
    from eye.features import load_gaze, adaptive_threshold, extract, _runs
    rows = []
    for cls in ('AD', 'NC'):
        for sub in sorted((Path(DATA) / cls).iterdir()):
            if not sub.is_dir():
                continue
            for task in TASKS:
                t, xy = load_gaze(sub / f'{task}.txt')
                with np.errstate(all='ignore'):
                    x = np.nanmean(xy[:, [0, 2]], axis=1); y = np.nanmean(xy[:, [1, 3]], axis=1)
                ok = np.isfinite(x) & np.isfinite(y)
                r = dict(cls=cls, subject=sub.name, task=task, n_rows=len(t), n_valid=int(ok.sum()),
                         dur_s=round((t[-1] - t[0]) / 1000, 1) if len(t) > 1 else 0.0)
                if ok.sum() >= 4:
                    tt, xx, yy = t[ok], x[ok], y[ok]
                    v = np.hypot(np.diff(xx), np.diff(yy)) / np.diff(tt)
                    pt = adaptive_threshold(v)
                    r.update(fs_hz=round(1000 / np.median(np.diff(t)), 0),
                             v_med=round(float(np.median(v)), 3), v_p99=round(float(np.percentile(v, 99)), 3),
                             v_max=round(float(v.max()), 2), pt=round(float(pt), 2),
                             pt_above_max=bool(pt >= v.max()), n_sac=len(_runs(v > pt)))
                rows.append(r)
    chk = pd.DataFrame(rows)
    pd.set_option('display.width', 250, 'display.max_rows', 100)
    print(chk.drop(columns=['cls']).to_string(index=False))
    print()
    print('recordings:', len(chk), '| with threshold above max velocity:', int(chk.get('pt_above_max', pd.Series(dtype=bool)).sum()),
          '| with 0 saccades:', int((chk.n_sac == 0).sum()))


**SEND BACK TO CLAUDE:** the full table and the last line printed by the cell above.

## Export to ONNX (flag / not flag only)
Fits the same pipeline on **all 20 subjects** and writes `eye_flag.onnx` plus `eye_features.json`.
This is for the app to run; it is **not** an evaluation (the 20 subjects are its training data). The pilot result above is still the only accuracy statement, and it is inconclusive.
The ONNX file holds only imputer + scaler + logistic regression. Turning gaze files into the 30 features (`eye/features.py`) runs in the app as Python.

In [ ]:
with log.tee('ONNX export'):
    import json, subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'skl2onnx', 'onnxruntime'], check=True)
    from sklearn.linear_model import LogisticRegression
    from skl2onnx import to_onnx
    import onnxruntime as ort
    from eye.cv import make_model

    os.makedirs('/content/out', exist_ok=True)
    X = df[cols].values.astype(np.float32); y = df.label.values
    final = make_model().fit(X, y)
    onx = to_onnx(final, X[:1], options={LogisticRegression: {'zipmap': False}}, target_opset=15)
    open('/content/out/eye_flag.onnx', 'wb').write(onx.SerializeToString())
    json.dump({'feature_order': cols, 'label_1': 'flagged (AD-like)', 'label_0': 'not flagged',
               'cutoff': 0.5, 'status': 'pilot n=20, inconclusive, webcam transfer unvalidated'},
              open('/content/out/eye_features.json', 'w'), indent=1)

    # parity: ONNX must give the same flag as sklearn, including rows with missing features
    sess = ort.InferenceSession('/content/out/eye_flag.onnx')
    onnx_flag = sess.run(None, {sess.get_inputs()[0].name: X})[0].ravel()
    sk_flag = final.predict(X)
    print('ONNX == sklearn flags on', int((onnx_flag == sk_flag).sum()), 'of', len(X), '| rows with NaN features:', int(np.isnan(X).any(axis=1).sum()))
    print('flagged on training subjects:', int(sk_flag.sum()), 'of', len(X), '(training data, not an accuracy)')


In [ ]:
# One zip = one download (browsers block several downloads at once)
import shutil
from google.colab import files
shutil.make_archive('/content/eye_results', 'zip', '/content/out')
print('Click Allow if the browser asks about downloads. If nothing downloads: Files panel > refresh > right-click eye_results.zip > Download.')
files.download('/content/eye_results.zip')


> **SEND BACK TO CLAUDE:** the two printed lines (parity count). Unzip `eye_results.zip`; save the `.onnx`/`.json` files to `D:\Alzheimers\models\` and `eye.log` to `D:\Alzheimersesults\`.
